# Phenology database — Parcelas-CL x Landsat

Built by `scripts/01_build_subset.py` -> `02_extract_phenology.py` -> `03_extract_topography.py`.

Unit of analysis: **one vegetation plot**, carrying a 52-step `PhenoShape` curve and the 18
land-surface-phenology (LSP) metrics derived from a **causal 3-year window** (`y-2..y`,
where `y` is the census year), over a 5x5 window of 30 m Landsat pixels.

Subset: 1,082 plots in central Chile (30-38S), 2003-2026, after removing plots with
duplicated coordinates (`md001` held 8 sites x 20 plots sharing a single coordinate).

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

DERIVED = Path('../data/derived')
PHENO = DERIVED / 'phenology'

plots = pd.read_parquet(DERIVED / 'plots_subset.parquet')
manifest = pd.read_csv(PHENO / 'manifest.csv')
topo = pd.read_parquet(DERIVED / 'topography' / 'topography.parquet')

# The manifest repeats columns that already come from plots_subset (metadata_id,
# Location, lat, lon). Without dropping them the merge renames both sides to _x/_y
# and the original name stops existing downstream.
dup = [c for c in ('metadata_id', 'Location', 'lat', 'lon') if c in manifest.columns]
db = (plots.merge(manifest.drop(columns=dup), left_on='PlotObservationID',
                  right_on='plot_id', how='inner')
           .merge(topo, on='plot_id', how='left'))

print(f'plots with phenology: {len(db)} of {len(plots)}')
if len(db) < len(plots):
    print('  extraction still running -> manifest is incomplete')
db.head()

## 1. Predictor quality

The largest gap along the DOY axis matters more than the raw observation count: 25
observations concentrated in summer do not constrain SOS. In Mediterranean Chile cloud
cover concentrates precisely at the start of the growing season, so **the missing data are
not random with respect to the metric being estimated**.

In [2]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(db['n_obs'], bins=40, color='#4C78A8')
ax[0].set(xlabel='Clear observations in the 3-year window', ylabel='Number of plots')
ax[1].hist(db['max_doy_gap'].dropna(), bins=40, color='#F58518')
ax[1].axvline(45, color='crimson', ls='--', label='45-day threshold')
ax[1].set(xlabel='Maximum DOY gap (days)', ylabel='Number of plots')
ax[1].legend()
sc = ax[2].scatter(db['n_obs'], db['max_doy_gap'], c=db['Year'], s=12, cmap='viridis')
ax[2].set(xlabel='Number of observations', ylabel='Maximum DOY gap (days)')
plt.colorbar(sc, ax=ax[2], label='Census year')
plt.tight_layout()

print(db.groupby('Year')[['n_obs', 'max_doy_gap']].median().round(1).to_string())

## 2. PhenoShape curves

Each `.nc` file stores `phenoshape` with dims `(index, doy, y, x)` — **5 vegetation
indices** x 52 steps x 5x5 pixels — plus the raw observations for both the indices and the
six scaled Landsat bands (`obs_*`). The bands are kept so any further index (MSAVI, NDMI,
NIRv) is a local computation rather than another pass over S3: SAVI could not be recovered
from a stored NDVI, which is the mistake that motivated keeping them.

Index roles differ, and that matters for the benchmark:

- `ndvi`, `kndvi`, `evi`, `savi` — greenness. `savi` carries the Huete (1988) soil
  adjustment (L = 0.5), which matters where sclerophyll matorral leaves a large exposed-soil
  fraction inside a 30 m pixel and NDVI both saturates and picks up a soil-brightness bias.
- `nbr` — moisture / burn, **not** greenness. It peaks at a different time of year, so its
  LSP metrics under a greenness-derived phase frame are not directly comparable.

In [3]:
def load_plot(plot_id):
    return xr.open_dataset(PHENO / f'{plot_id}.nc')

def centre_curve(ds, index='ndvi'):
    c = ds['phenoshape'].sel(index=index)
    return c.isel(y=c.sizes['y'] // 2, x=c.sizes['x'] // 2)

example = db['plot_id'].iloc[0]
ds = load_plot(example)
INDICES = [str(v) for v in np.atleast_1d(ds['index'].values)]
print('indices stored:', INDICES)
print('bands stored  :', sorted(v for v in ds.data_vars if v.startswith('obs_band_')))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for idx in INDICES:
    ax[0].plot(ds.doy, centre_curve(ds, idx), label=idx.upper())
ax[0].set(title=f'Plot {example}: curves by vegetation index',
          xlabel='Day of year', ylabel='Index value')
ax[0].legend()

# Spread across the 25 pixels: spectral mixing within the plot footprint
cur = ds['phenoshape'].sel(index='ndvi').stack(px=('y', 'x'))
ax[1].plot(ds.doy, cur.values, color='grey', alpha=.35, lw=.8)
ax[1].plot(ds.doy, cur.mean('px'), color='crimson', lw=2, label='5x5 mean')
ax[1].set(title='NDVI: variability across the 25 pixels',
          xlabel='Day of year', ylabel='NDVI')
ax[1].legend()
plt.tight_layout()

In [4]:
# Curves for every index and plot, as a tidy frame: {index: DataFrame(plot x 52 steps)}
curves = {ix: {} for ix in INDICES}
for pid in db['plot_id']:
    f = PHENO / f'{pid}.nc'
    if not f.exists():
        continue
    with xr.open_dataset(f) as d:
        for ix in np.atleast_1d(d['index'].values):
            curves[str(ix)][pid] = centre_curve(d, str(ix)).values

C = {ix: pd.DataFrame(v).T for ix, v in curves.items() if v}
for ix, t in C.items():
    print(f'{ix:6s} {t.shape[0]} plots x {t.shape[1]} steps')

# Mean curve per index: how much do the indices actually differ in shape?
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for ix, t in C.items():
    ax[0].plot(np.arange(t.shape[1]), t.mean(), label=ix.upper())
ax[0].set(xlabel='Curve step (52 ~ weekly)', ylabel='Index value',
          title='Mean curve by vegetation index')
ax[0].legend()

# Correlation between indices, on the curve: a redundant index adds nothing to the benchmark
flat = pd.DataFrame({ix: t.loc[sorted(set.intersection(*(set(u.index) for u in C.values())))]
                       .to_numpy().ravel() for ix, t in C.items()})
corr = flat.corr()
im = ax[1].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax[1].set_xticks(range(len(corr)), [c.upper() for c in corr.columns], rotation=45)
ax[1].set_yticks(range(len(corr)), [c.upper() for c in corr.columns])
for i in range(len(corr)):
    for j in range(len(corr)):
        ax[1].text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
ax[1].set_title('Between-index correlation (all curve points)')
plt.colorbar(im, ax=ax[1])
plt.tight_layout()

## 3. Land-surface-phenology metrics

18 metrics per vegetation index. Two diagnostics to read before trusting any of them:

- `sos == pos` marks degenerate season detection.
- **`order_coherent`** marks whether the season actually runs SOS -> POS -> EOS (one wrap
  of the year allowed). This catches mis-anchoring that the `sos == pos` test misses
  entirely: a season anchored in the wrong phase frame can have `sos != pos` and still be
  nonsense, e.g. a peak that precedes its own start.

The consolidated table comes from `scripts/04_recompute_lsp.py --index all`, which also
records `phase_group` (summer- vs winter-peaking) and the anchoring actually used.

In [5]:
# Consolidated LSP table: one row per plot x index. Produced by
#   python scripts/04_recompute_lsp.py --hemisphere by_phase --index all
LSP_TABLE = DERIVED / 'lsp_all_by_phase.parquet'
LSP = ['sos', 'pos', 'eos', 'los', 'ampl', 'vpos', 'vsos', 'veos', 'rog', 'ros', 'trough', 'msp']

if LSP_TABLE.exists():
    lsp_long = pd.read_parquet(LSP_TABLE)
else:
    print(f'{LSP_TABLE.name} not found -- run scripts/04_recompute_lsp.py first;'
          '\nfalling back to the lsp_* variables stored inside each .nc')
    rows = []
    for pid in db['plot_id']:
        f = PHENO / f'{pid}.nc'
        if not f.exists():
            continue
        with xr.open_dataset(f) as d:
            for ix in np.atleast_1d(d['index'].values):
                r = {'plot_id': pid, 'index': str(ix)}
                for m in LSP:
                    v = f'lsp_{m}'
                    if v in d:
                        a = d[v].sel(index=ix)
                        r[m] = float(a.isel(y=a.sizes['y'] // 2, x=a.sizes['x'] // 2).values)
                rows.append(r)
    lsp_long = pd.DataFrame(rows)

print('rows:', len(lsp_long), '| plots:', lsp_long['plot_id'].nunique(),
      '| indices:', sorted(lsp_long['index'].unique()))
if 'order_coherent' in lsp_long:
    print('\nseason order coherence by index:')
    print((100 * lsp_long.groupby('index')['order_coherent'].mean()).round(1).to_string())
print('\nmedian metrics by index:')
print(lsp_long.groupby('index')[['sos', 'pos', 'eos', 'los', 'ampl']].median().round(1).to_string())

In [6]:
# Distribution of each metric, one line per index
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for a, m in zip(axes.ravel(), LSP):
    if m not in lsp_long:
        continue
    for ix, g in lsp_long.groupby('index'):
        v = g[m].replace([np.inf, -np.inf], np.nan).dropna()
        if len(v):
            a.hist(v, bins=30, histtype='step', lw=1.4, label=ix.upper())
    a.set(title=m.upper(), xlabel='Value', ylabel='Plots')
axes.ravel()[0].legend(fontsize=7)
fig.suptitle('LSP metric distributions by vegetation index (centre pixel)', y=1.01)
plt.tight_layout()

## 4. Spatial distribution and topography

In [7]:
# Wide table: one row per plot, LSP columns suffixed by index -- the shape a model wants
wide = lsp_long.pivot_table(index='plot_id', columns='index',
                            values=[m for m in LSP if m in lsp_long])
wide.columns = [f'{m}_{ix}' for m, ix in wide.columns]
d = db.merge(wide, left_on='plot_id', right_index=True, how='left')
print('model table:', d.shape)

fig, ax = plt.subplots(1, 3, figsize=(16, 6))
for a, (col, lab) in zip(ax, [('elevation', 'Elevation (m)'),
                              ('richness', 'Species richness'),
                              ('los_ndvi', 'Length of season, NDVI (days)')]):
    if col not in d:
        continue
    s = a.scatter(d['lon'], d['lat'], c=d[col], s=14, cmap='viridis')
    plt.colorbar(s, ax=a, label=lab)
    a.set(xlabel='Longitude', ylabel='Latitude')
fig.suptitle('Plot distribution across central Chile', y=1.0)
plt.tight_layout()

In [8]:
# Does phenology respond to topography, and does the answer depend on the index?
# If nothing shows up here, that is worth knowing before training any model.
topo_cols = ['elevation', 'slope', 'northness', 'heat_load', 'tpi']
rows = []
for ix in sorted(lsp_long['index'].unique()):
    cols = [f'{m}_{ix}' for m in ['sos', 'pos', 'eos', 'los', 'ampl'] if f'{m}_{ix}' in d]
    sub = d[topo_cols + cols + ['richness']].replace([np.inf, -np.inf], np.nan)
    c = sub.corr(method='spearman')
    for m in cols:
        rows.append({'index': ix, 'metric': m.rsplit('_', 1)[0],
                     **{t: c.loc[m, t] for t in topo_cols}, 'richness': c.loc[m, 'richness']})
H = pd.DataFrame(rows).set_index(['index', 'metric'])

fig, ax = plt.subplots(figsize=(8, max(4, 0.32 * len(H))))
im = ax.imshow(H, cmap='RdBu_r', vmin=-.6, vmax=.6, aspect='auto')
ax.set_xticks(range(H.shape[1]), H.columns, rotation=45, ha='right')
ax.set_yticks(range(len(H)), [f'{a}/{b}' for a, b in H.index], fontsize=7)
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        if np.isfinite(H.iloc[i, j]):
            ax.text(j, i, f'{H.iloc[i, j]:.2f}', ha='center', va='center', fontsize=6)
plt.colorbar(im, label='Spearman rho')
ax.set_title('LSP x topography x richness, per index')
plt.tight_layout()

## 5. Cross-validation folds

Never random CV. Most source projects are **single-year**, so leave-one-dataset-out is
simultaneously leave-one-year-out and leave-one-protocol-out: the strictest transferability
test available in this dataset.

In [9]:
cv = pd.read_parquet(DERIVED / 'cv_folds.parquet')
groups = pd.read_csv(DERIVED / 'groups_dataset.csv')
print(groups.head(15).to_string(index=False))

for scheme in cv['scheme'].unique():
    s = cv[cv['scheme'] == scheme]
    sizes = s[s['split'] == 'test'].groupby('fold').size()
    print(f'\n{scheme}: {s["fold"].nunique()} folds, test n = {sizes.tolist()}')

## 6. Export the paper figure set

This notebook is for exploring. **`scripts/06_paper_figures.py` is the single source of
truth for anything that goes into the paper** — running it from here rather than
re-plotting means the exported figure is exactly the one reviewed here, and there is no
second copy to drift out of sync.

Format policy, applied automatically by content:

- **PDF** for vector figures (lines, points, polygons, text). `pdf.fonttype = 42` keeps text
  as searchable, restylable text rather than outlines.
- **PNG at 300 dpi** for figures with genuine pixel imagery — only `figS1`, which shows the
  8x8 reshape used as the CNN substrate.

Point maps use `shapefiles/regiones_chile.shp`, clipped to the study window and simplified
to ~500 m before drawing. Without that clip the full-resolution southern fjords stay in the
file even though `xlim` hides them, which made the first draft of Fig. 1 weigh 16 MB
instead of 0.5 MB.

In [ ]:
import subprocess, sys
from pathlib import Path

OUT = Path('../results/figures')
r = subprocess.run([sys.executable, '../scripts/06_paper_figures.py', '--out-dir', str(OUT)],
                   capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode:
    print('STDERR:', r.stderr[-2000:])

for f in sorted(OUT.glob('*')):
    print(f'{f.name:34s} {f.stat().st_size/1024:8.0f} KB')

In [ ]:
# Preview a rendered figure inline without leaving the notebook
from IPython.display import Image, display

png = OUT / 'figS1_example_plot.png'
if png.exists():
    display(Image(filename=str(png), width=900))
else:
    print('run the export cell first')